# Notebook 02 — Goodreads Paper Feature Engineering



This notebook starts from your already-filtered `filtered_reviews.csv` in Google Drive. It does **not** download or reconstruct the raw Goodreads corpus.

## Features implemented from the paper/repository
- `rating_diff = user_rating - avg_rating`
- quotation indicator
- number of sentences and words
- average sentence length and average word length
- percentage of verbs, nouns, and adjective/adverb tags (the repository combines adjectives and adverbs as `pct_adj`)
- VADER compound sentiment averaged over sentences
- NLTK stopword removal and WordNet lemmatization
- tokenized review text for BOW and TF-IDF

The existing `popular` target is retained, not silently recalculated. Confirm it was created with the paper's popularity-share cutoff (the paper reports a 0.02 cutoff) before describing results as an exact reproduction.






## 1. Install dependencies

In [2]:
%pip -q install pandas numpy nltk tqdm pyarrow

## 2. Mount Drive and locate `filtered_reviews.csv`

In [3]:
from pathlib import Path
import os, json, time, re, math, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')
candidate_paths = [
    MYDRIVE / 'filtered_reviews .csv',
    MYDRIVE / 'goodreads' / 'filtered_reviews .csv',
    MYDRIVE / 'data' / 'filtered_reviews .csv',
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    matches = list(MYDRIVE.rglob('filtered_reviews.csv'))
    if matches:
        DATA_PATH = matches[0]

if DATA_PATH is None:
    raise FileNotFoundError(
        'Could not find filtered_reviews.csv in MyDrive. '
        'Move it to MyDrive root or update DATA_PATH manually.'
    )

OUTPUT_DIR = MYDRIVE / 'goodreads_reproduction_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURED_PATH = OUTPUT_DIR / 'goodreads_paper_features.csv'

print('Input:', DATA_PATH)
print('Output:', FEATURED_PATH)
print('Input size (GB):', round(DATA_PATH.stat().st_size / 1024**3, 3))

Mounted at /content/drive
Input: /content/drive/MyDrive/filtered_reviews .csv
Output: /content/drive/MyDrive/goodreads_reproduction_results/goodreads_paper_features.csv
Input size (GB): 2.194


## 3. Inspect input schema and configure run

In [4]:
# Change to None for all rows. Use e.g. 5000 for a quick smoke test.
MAX_ROWS = None
CHUNK_SIZE = 5000
RANDOM_STATE = 229

header = pd.read_csv(DATA_PATH, nrows=5)
print('Columns:', header.columns.tolist())
display(header.head())

REQUIRED = {
    'user_reviews', 'user_rating', 'avg_rating',
    'review_text', 'days_since_review', 'popular'
}
missing = REQUIRED - set(header.columns)
if missing:
    raise ValueError(
        f'Missing required columns: {sorted(missing)}. '
        f'Found: {header.columns.tolist()}'
    )

if 'popular' not in header.columns:
    raise ValueError("The existing 'popular' target is required.")

Columns: ['book_id', 'user_reviews', 'user_rating', 'avg_rating', 'ratings_count', 'review_text', 'days_since_review', 'review_likes', 'like_share', 'popular']


,book_id,user_reviews,user_rating,avg_rating,ratings_count,review_text,days_since_review,review_likes,like_share,popular
0,18245960,218,5,4.01,6336.0,This is a special book. It started slow for ab...,96,29.0,0.145729,1
1,29983426,218,5,3.90,439.0,I decided to give up eating processed sugar fo...,299,32.0,0.492308,1
2,28114110,218,4,4.00,880.0,"Kevin Kelly, who is a Wired co-founder, lays o...",327,14.0,0.191781,1
3,28684704,218,3,4.10,13990.0,"A fun, fast paced science fiction thriller. I ...",353,22.0,0.021782,1
4,28119237,218,4,4.04,10078.0,A fascinating book about community and belongi...,407,32.0,0.113074,1


## 4. Download NLTK resources (including newer Colab resource names)

In [5]:
import nltk

NLTK_RESOURCES = [
    ('tokenizers/punkt', 'punkt'),
    ('tokenizers/punkt_tab', 'punkt_tab'),
    ('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger'),
    ('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng'),
    ('corpora/stopwords', 'stopwords'),
    ('corpora/wordnet', 'wordnet'),
    ('corpora/omw-1.4', 'omw-1.4'),
    ('sentiment/vader_lexicon', 'vader_lexicon'),
]
for resource_path, package_name in NLTK_RESOURCES:
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package_name, quiet=True)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

sid = SentimentIntensityAnalyzer()
lemmatizer = WordNetLemmatizer()
STOPWORDS = set(stopwords.words('english'))

print('NLTK resources ready.')

NLTK resources ready.


## 5. Define paper-aligned feature functions

In [6]:
def safe_sent_tokenize(text):
    text = '' if pd.isna(text) else str(text)
    if not text.strip():
        return []
    try:
        return nltk.tokenize.sent_tokenize(text)
    except LookupError:
        # Defensive fallback for a broken/missing tokenizer resource.
        parts = re.split(r'(?<=[.!?])\\s+', text.strip())
        return [p for p in parts if p]

def safe_word_tokenize(text):
    text = '' if pd.isna(text) else str(text)
    if not text.strip():
        return []
    try:
        tokens = nltk.tokenize.word_tokenize(text)
    except LookupError:
        tokens = nltk.tokenize.TreebankWordTokenizer().tokenize(text)
    return [w.lower() for w in tokens if w.isalpha()]

def count_prefix(tags, prefixes):
    return sum(1 for _, tag in tags if tag and tag[0] in prefixes)

def engineer_one_review(row):
    raw = '' if pd.isna(row.get('review_text')) else str(row.get('review_text'))
    sentences = safe_sent_tokenize(raw)
    words = safe_word_tokenize(raw)

    # POS counts are computed on alphabetic lowercase words before stopword removal,
    # matching the ordering in the repository's features.py.
    try:
        pos_tags = nltk.pos_tag(words) if words else []
    except LookupError:
        # A resource error should not crash a multi-million-row batch.
        pos_tags = [(w, 'NN') for w in words]

    n_sent = len(sentences)
    n_words = len(words)
    n_letters = sum(len(w) for w in words)

    verbs = count_prefix(pos_tags, ['V'])
    nouns = count_prefix(pos_tags, ['N'])
    # The repository groups adjective and adverb tags into pct_adj.
    adj_adv = count_prefix(pos_tags, ['J', 'R'])

    sentiment = (
        float(np.mean([sid.polarity_scores(s)['compound'] for s in sentences]))
        if sentences else 0.0
    )

    clean_words = [w for w in words if w not in STOPWORDS]
    clean_words = [lemmatizer.lemmatize(w) for w in clean_words]

    try:
        rating_diff = float(row['user_rating']) - float(row['avg_rating'])
    except (TypeError, ValueError):
        rating_diff = np.nan

    return {
        'rating_diff': rating_diff,
        'quote': int('"' in raw),
        'num_words': n_words,
        'avg_sent_len': (n_words / n_sent) if n_sent else 0.0,
        'avg_word_len': (n_letters / n_words) if n_words else 0.0,
        'pct_verbs': (verbs / n_words) if n_words else 0.0,
        'pct_nouns': (nouns / n_words) if n_words else 0.0,
        'pct_adj': (adj_adv / n_words) if n_words else 0.0,
        'sentiment': sentiment,
        'tokenized_words': ' '.join(clean_words),
    }

# Small smoke test before processing the entire file.
test_row = header.iloc[0].to_dict()
test_features = engineer_one_review(test_row)
print(test_features)
assert set([
    'rating_diff', 'quote', 'num_words', 'avg_sent_len',
    'avg_word_len', 'pct_verbs', 'pct_nouns', 'pct_adj',
    'sentiment', 'tokenized_words'
]).issubset(test_features)

{'rating_diff': 0.9900000000000002, 'quote': 1, 'num_words': 354, 'avg_sent_len': 17.7, 'avg_word_len': 4.61864406779661, 'pct_verbs': 0.192090395480226, 'pct_nouns': 0.22598870056497175, 'pct_adj': 0.1638418079096045, 'sentiment': 0.12364999999999997, 'tokenized_words': 'special book started slow first third middle third started get interesting last third blew mind love good science fiction push thinking thing go hugo winner translated original chinese made interesting different way thing read instance intermixing chinese revolutionary history kept accusing people reactionary etc book science alien science described book impressive book grounded physic pretty accurate far could tell though got folding proton dimension think making stuff interesting think though would happen seti station received message found someone person monitoring answering signal side disillusioned part book bit dark would like think human reaction discovering alien civilization hostile would like enders game wou

## 6. Process the CSV in resumable chunks and save checkpoints

Each input chunk is saved as its own checkpoint CSV in Google Drive. If Colab stops, rerun this cell: completed checkpoints are skipped. After all chunks finish, they are merged into `goodreads_paper_features.csv`. Do not manually delete the checkpoint folder during processing.


In [7]:
# Resume-safe processing:
# - One atomic checkpoint file per input chunk.
# - Completed chunks are skipped after a runtime restart.
# - A .tmp file is never considered complete.
# - The final combined CSV is rebuilt only after all requested chunks exist.
import gc

RUN_TAG = 'all_rows' if MAX_ROWS is None else f'first_{MAX_ROWS}_rows'
CHECKPOINT_DIR = OUTPUT_DIR / f'feature_chunk_checkpoints_{RUN_TAG}'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = OUTPUT_DIR / f'feature_engineering_progress_{RUN_TAG}.json'

# Identify the exact input/config so checkpoints from another run are not mixed in.
run_signature = {
    'input_path': str(DATA_PATH),
    'input_size_bytes': int(DATA_PATH.stat().st_size),
    'input_mtime': int(DATA_PATH.stat().st_mtime),
    'chunk_size': int(CHUNK_SIZE),
    'max_rows': MAX_ROWS,
}
if PROGRESS_PATH.exists():
    old_progress = json.loads(PROGRESS_PATH.read_text())
    old_signature = old_progress.get('run_signature', {})
    if old_signature != run_signature:
        raise RuntimeError(
            'The saved checkpoints were created with a different input file or run settings. '
            'To start a fresh run, rename/delete feature_chunk_checkpoints and '
            'feature_engineering_progress.json, then rerun this cell.'
        )
else:
    existing_checkpoints = list(CHECKPOINT_DIR.glob('chunk_*.csv'))
    if existing_checkpoints:
        raise RuntimeError(
            'Checkpoint CSVs exist but the progress manifest is missing. '
            'For safety, do not overwrite them automatically. Restore the manifest or '
            'rename/delete the checkpoint folder to start fresh.'
        )
    PROGRESS_PATH.write_text(json.dumps({
        'run_signature': run_signature,
        'completed_chunks': [],
        'rows_written_per_chunk': {},
    }, indent=2))

progress = json.loads(PROGRESS_PATH.read_text())
completed_chunks = set(int(x) for x in progress.get('completed_chunks', []))
rows_written_per_chunk = {
    int(k): int(v) for k, v in progress.get('rows_written_per_chunk', {}).items()
}

start_time = time.time()
processed_this_session = 0
source_rows_seen = 0
all_requested_chunks = 0

reader = pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE, low_memory=False)
for chunk_number, chunk in enumerate(reader, start=1):
    # Enforce MAX_ROWS by source rows, not by rows remaining after target filtering.
    if MAX_ROWS is not None:
        remaining = MAX_ROWS - source_rows_seen
        if remaining <= 0:
            break
        if len(chunk) > remaining:
            chunk = chunk.iloc[:remaining].copy()
    source_rows_seen += len(chunk)
    all_requested_chunks = chunk_number

    checkpoint_path = CHECKPOINT_DIR / f'chunk_{chunk_number:06d}.csv'
    if chunk_number in completed_chunks and checkpoint_path.exists():
        # A checkpoint is only trusted when both manifest and file exist.
        continue
    if checkpoint_path.exists() and chunk_number not in completed_chunks:
        # A crash could leave a renamed file before manifest update; validate it and adopt it.
        try:
            checkpoint_rows = sum(1 for _ in open(checkpoint_path, 'r', encoding='utf-8')) - 1
            rows_written_per_chunk[chunk_number] = max(0, checkpoint_rows)
            completed_chunks.add(chunk_number)
            progress['completed_chunks'] = sorted(completed_chunks)
            progress['rows_written_per_chunk'] = {str(k): v for k, v in rows_written_per_chunk.items()}
            PROGRESS_PATH.write_text(json.dumps(progress, indent=2))
            continue
        except Exception:
            checkpoint_path.unlink(missing_ok=True)

    if MAX_ROWS is not None and source_rows_seen > MAX_ROWS:
        raise RuntimeError('Internal MAX_ROWS accounting error.')

    chunk['popular'] = pd.to_numeric(chunk['popular'], errors='coerce')
    chunk = chunk[chunk['popular'].isin([0, 1])].copy()

    if not chunk.empty:
        engineered = []
        for record in tqdm(chunk.to_dict(orient='records'), desc=f'Chunk {chunk_number}', leave=False):
            engineered.append(engineer_one_review(record))
        feat_df = pd.DataFrame(engineered, index=chunk.index)
        out = pd.concat([chunk.reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)
    else:
        # Keep an empty but correctly headed checkpoint if a chunk has no valid targets.
        out = chunk.copy()
        for col in [
            'rating_diff', 'quote', 'num_words', 'avg_sent_len', 'avg_word_len',
            'pct_verbs', 'pct_nouns', 'pct_adj', 'sentiment', 'tokenized_words'
        ]:
            out[col] = pd.Series(dtype='object')

    tmp_path = checkpoint_path.with_suffix('.csv.tmp')
    out.to_csv(tmp_path, index=False)
    os.replace(tmp_path, checkpoint_path)  # atomic rename on the same Drive mount

    rows_written_per_chunk[chunk_number] = int(len(out))
    completed_chunks.add(chunk_number)
    progress['completed_chunks'] = sorted(completed_chunks)
    progress['rows_written_per_chunk'] = {str(k): v for k, v in rows_written_per_chunk.items()}
    progress['last_completed_chunk'] = int(chunk_number)
    progress['updated_at_unix'] = time.time()
    PROGRESS_PATH.write_text(json.dumps(progress, indent=2))
    processed_this_session += len(out)

    if chunk_number % 5 == 0:
        elapsed = time.time() - start_time
        print(
            f'Input chunk {chunk_number}: this session processed {processed_this_session:,} rows '
            f'in {elapsed/60:.1f} min. Checkpoints complete: {len(completed_chunks):,}.'
        )
    del chunk, out
    if 'engineered' in locals():
        del engineered
    if 'feat_df' in locals():
        del feat_df
    gc.collect()

if all_requested_chunks == 0:
    raise RuntimeError('No input rows were found.')

expected_chunks = set(range(1, all_requested_chunks + 1))
missing_chunks = sorted(expected_chunks - completed_chunks)
if missing_chunks:
    raise RuntimeError(
        f'Not all chunks are complete. Missing {len(missing_chunks)} checkpoint(s); '
        f'first missing: {missing_chunks[:10]}. Rerun this cell to resume.'
    )

# Merge checkpoint files into one CSV for Notebook 03. This merge can be rerun safely.
merge_tmp = FEATURED_PATH.with_suffix('.csv.tmp')
if merge_tmp.exists():
    merge_tmp.unlink()
first_write = True
merged_rows = 0
for chunk_number in tqdm(range(1, all_requested_chunks + 1), desc='Merging checkpoints'):
    checkpoint_path = CHECKPOINT_DIR / f'chunk_{chunk_number:06d}.csv'
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')
    part = pd.read_csv(checkpoint_path, low_memory=False)
    part.to_csv(merge_tmp, mode='w' if first_write else 'a', header=first_write, index=False)
    first_write = False
    merged_rows += len(part)
    del part
    gc.collect()

if first_write:
    raise RuntimeError('No checkpoint data was merged.')
os.replace(merge_tmp, FEATURED_PATH)
progress['merge_complete'] = True
progress['merged_rows'] = int(merged_rows)
progress['expected_chunks'] = int(all_requested_chunks)
PROGRESS_PATH.write_text(json.dumps(progress, indent=2))

if merged_rows == 0:
    raise RuntimeError('No valid rows were processed. Check the target and input file.')
print(f'Finished/resumed successfully: {merged_rows:,} rows across {all_requested_chunks:,} chunks.')
print('Combined output:', FEATURED_PATH)
print('Checkpoint folder:', CHECKPOINT_DIR)
print('Progress manifest:', PROGRESS_PATH)
print('Time spent in this session (minutes):', round((time.time() - start_time) / 60, 2))


Merging checkpoints:   0%|          | 0/397 [00:00<?, ?it/s]

Finished/resumed successfully: 1,981,129 rows across 397 chunks.
Combined output: /content/drive/MyDrive/goodreads_reproduction_results/goodreads_paper_features.csv
Checkpoint folder: /content/drive/MyDrive/goodreads_reproduction_results/feature_chunk_checkpoints_all_rows
Progress manifest: /content/drive/MyDrive/goodreads_reproduction_results/feature_engineering_progress_all_rows.json
Time spent in this session (minutes): 11.22


## 7. Validate the engineered dataset

In [8]:
featured_head = pd.read_csv(FEATURED_PATH, nrows=10)
print('Output columns:', featured_head.columns.tolist())
display(featured_head[[
    c for c in [
        'popular', 'user_reviews', 'days_since_review', 'user_rating',
        'rating_diff', 'num_words', 'avg_word_len', 'avg_sent_len',
        'pct_verbs', 'pct_nouns', 'pct_adj', 'quote', 'sentiment',
        'tokenized_words'
    ] if c in featured_head.columns
]].head())

# Count rows and validate binary target.
target_series = pd.read_csv(FEATURED_PATH, usecols=['popular'])['popular']
row_count = int(len(target_series))
target_counts = target_series.value_counts().to_dict()
manifest = {
    'input_path': str(DATA_PATH),
    'output_path': str(FEATURED_PATH),
    'rows_written': int(row_count),
    'target_counts': {str(k): int(v) for k, v in target_counts.items()},
    'features_added': [
        'rating_diff', 'quote', 'num_words', 'avg_sent_len', 'avg_word_len',
        'pct_verbs', 'pct_nouns', 'pct_adj', 'sentiment', 'tokenized_words'
    ],
    'max_rows_setting': MAX_ROWS,
    'random_state': RANDOM_STATE,
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'progress_manifest': str(PROGRESS_PATH),
    'completed_chunks': len(progress.get('completed_chunks', [])),
    'expected_chunks': progress.get('expected_chunks')
}
with open(OUTPUT_DIR / 'feature_engineering_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))


Output columns: ['book_id', 'user_reviews', 'user_rating', 'avg_rating', 'ratings_count', 'review_text', 'days_since_review', 'review_likes', 'like_share', 'popular', 'rating_diff', 'quote', 'num_words', 'avg_sent_len', 'avg_word_len', 'pct_verbs', 'pct_nouns', 'pct_adj', 'sentiment', 'tokenized_words']


,popular,user_reviews,days_since_review,user_rating,rating_diff,num_words,avg_word_len,avg_sent_len,pct_verbs,pct_nouns,pct_adj,quote,sentiment,tokenized_words
0,1,218,96,5,0.99,354,4.618644,17.700000,0.192090,0.225989,0.163842,1,0.123650,special book started slow first third middle t...
1,1,218,299,5,1.10,966,4.623188,21.466667,0.180124,0.236025,0.187371,1,0.030033,decided give eating processed sugar month janu...
2,1,218,327,4,0.00,567,4.414462,20.250000,0.186949,0.222222,0.185185,1,0.232521,kevin kelly wired lay technological trend inev...
3,1,218,353,3,-1.10,458,4.303493,13.085714,0.209607,0.235808,0.120087,1,0.096211,fun fast paced science fiction thriller read n...
4,1,218,407,4,-0.04,494,4.757085,27.444444,0.155870,0.275304,0.147773,1,0.047972,fascinating book community belonging modern so...


{
  "input_path": "/content/drive/MyDrive/filtered_reviews .csv",
  "output_path": "/content/drive/MyDrive/goodreads_reproduction_results/goodreads_paper_features.csv",
  "rows_written": 1981129,
  "target_counts": {
    "0": 1572276,
    "1": 408853
  },
  "features_added": [
    "rating_diff",
    "quote",
    "num_words",
    "avg_sent_len",
    "avg_word_len",
    "pct_verbs",
    "pct_nouns",
    "pct_adj",
    "sentiment",
    "tokenized_words"
  ],
  "max_rows_setting": null,
  "random_state": 229,
  "checkpoint_dir": "/content/drive/MyDrive/goodreads_reproduction_results/feature_chunk_checkpoints_all_rows",
  "progress_manifest": "/content/drive/MyDrive/goodreads_reproduction_results/feature_engineering_progress_all_rows.json",
  "completed_chunks": 397,
  "expected_chunks": 397
}


## Notebook 02 completion checklist
- `goodreads_paper_features.csv` is the output for Notebook 03.
- `feature_engineering_manifest.json` records the processed row count.
- The `popular` target is carried forward as supplied; verify its cutoff before claiming exact reproduction.
- In the repository, the adjective/adverb POS feature is combined as `pct_adj`, so this notebook follows that implementation.